# Demo API -> Kaggle GPU -> Resultados

Este notebook es el cliente para la version del jurado. Sube un video a tu API, la API lanza un notebook Kaggle con GPU, luego consulta estado, hace pull y descarga artefactos.


## 1. Configuracion

Antes de ejecutar:

- arranca el servidor con `dvka-server`;
- configura Kaggle CLI en el servidor;
- coloca `models/best.pt` o configura `configs/kaggle_inference.json` con `model_dataset_slug`.


In [ ]:
from pathlib import Path
import time
import requests

API_URL = "http://localhost:8000"
VIDEO_PATH = Path(r"C:\\path\\to\\video.mp4")  # cambia esto
OUT_DIR = Path("api_downloads")
OUT_DIR.mkdir(exist_ok=True)


## 2. Healthcheck


In [ ]:
r = requests.get(f"{API_URL}/health", timeout=30)
r.raise_for_status()
r.json()


## 3. Enviar video y lanzar Kaggle


In [ ]:
assert VIDEO_PATH.exists(), VIDEO_PATH
with VIDEO_PATH.open("rb") as f:
    r = requests.post(
        f"{API_URL}/v1/videos/analyze-kaggle",
        files={"video": (VIDEO_PATH.name, f, "video/mp4")},
        timeout=300,
    )
r.raise_for_status()
job = r.json()
job


## 4. Consultar estado hasta que termine


In [ ]:
job_id = job["job_id"]
while True:
    r = requests.get(f"{API_URL}/v1/kaggle-jobs/{job_id}", timeout=60)
    r.raise_for_status()
    status = r.json()
    print(status.get("status"), status.get("kernel_id"))
    if status.get("status") in {"complete", "error"}:
        break
    time.sleep(30)
status


## 5. Hacer pull de resultados desde Kaggle al servidor


In [ ]:
r = requests.post(f"{API_URL}/v1/kaggle-jobs/{job_id}/pull", timeout=300)
r.raise_for_status()
pulled = r.json()
pulled


## 6. Descargar artefactos


In [ ]:
artifact_names = [
    "summary.json",
    "tracking_frame_detections.csv",
    "tracking_track_summary.csv",
    "tracking_class_votes.csv",
]

downloaded = []
for name in artifact_names:
    url = f"{API_URL}/v1/kaggle-jobs/{job_id}/artifacts/{name}"
    r = requests.get(url, timeout=120)
    if r.status_code == 404:
        print("missing", name)
        continue
    r.raise_for_status()
    out = OUT_DIR / f"{job_id}_{name}"
    out.write_bytes(r.content)
    downloaded.append(out)
downloaded


## 7. Ver resumen de tracks


In [ ]:
import pandas as pd

track_csv = OUT_DIR / f"{job_id}_tracking_track_summary.csv"
if track_csv.exists() and track_csv.stat().st_size > 0:
    df = pd.read_csv(track_csv)
    display(df.head())
    display(df["final_class"].value_counts())
else:
    print("No track summary downloaded")
